# Support Vector Classification (SVC)

This notebook demonstrates SVC classification using the Iris dataset. It preserves the original experiment; follow-up improvements will be added after the baseline has been reviewed.

In [ ]:
from sklearn.svm import SVC
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import datasets

In [ ]:
df = datasets.load_iris(as_frame=True).frame

In [ ]:
df.isnull().sum()
df.shape
df.head()

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Model
svc = SVC()
svc.fit(X_train_scaled, y_train)

In [ ]:
y_pred = svc.predict(X_test_scaled)

In [ ]:
# Evaluate
from sklearn.metrics import accuracy_score, classification_report

print("accuracy: ", accuracy_score(y_test, y_pred))
print("classification report: \n ", classification_report(y_test, y_pred))

## Confusion matrix

A confusion matrix shows which classes were predicted correctly and which were confused with one another.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_estimator(svc, X_test_scaled, y_test)
plt.title("SVC Confusion Matrix")
plt.tight_layout()
plt.show()

## RBF kernel

RBF is the default kernel for `SVC`, but this explicit experiment makes it easier to compare it with Linear, Polynomial, and Sigmoid kernels.

In [ ]:
# RBF kernel
svc = SVC(kernel="rbf")
svc.fit(X_train_scaled, y_train)
y_pred = svc.predict(X_test_scaled)
print("accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Linear kernel

svc = SVC(kernel="linear")
svc.fit(X_train_scaled, y_train)

y_pred = svc.predict(X_test_scaled)

print("accuracy: ", accuracy_score(y_test, y_pred))

In [ ]:
# Polynomial kernel

svc = SVC(kernel="poly")
svc.fit(X_train_scaled, y_train)

y_pred = svc.predict(X_test_scaled)

print("accuracy: ", accuracy_score(y_test, y_pred))

In [ ]:
# Sigmoid kernel

svc = SVC(kernel="sigmoid")
svc.fit(X_train_scaled, y_train)

y_pred = svc.predict(X_test_scaled)

print("accuracy: ", accuracy_score(y_test, y_pred))

## Exploratory comparison of C values

This loop is retained as an illustration of how `C` changes an SVC model. Because it reports test-set scores repeatedly, do not use this loop to select a final model; use cross-validation on the training data instead.

In [ ]:
C_vals = [0.5, 1, 2, 3, 4, 5]

for c_val in C_vals:
    svc = SVC(C=c_val, kernel="rbf")
    svc.fit(X_train_scaled, y_train)
    y_pred = svc.predict(X_test_scaled)
    print("C =", c_val, "test accuracy:", accuracy_score(y_test, y_pred))

## Hyperparameter tuning with GridSearchCV

This pipeline fits scaling inside each cross-validation fold, preventing validation-fold information from being used to fit the scaler.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

svc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

param_grid = {
    "svc__C": [0.5, 1, 2, 3, 4, 5],
    "svc__kernel": ["linear", "poly", "rbf", "sigmoid"]
}

grid_search = GridSearchCV(
    svc_pipeline,
    param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)
print("Held-out test accuracy:", grid_search.best_estimator_.score(X_test, y_test))